# Wav2vec: Pretrained

In [1]:
!pip install datasets transformers
!pip install https://github.com/kpu/kenlm/archive/master.zip pyctcdecode


  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from transformers import Wav2Vec2ProcessorWithLM
processor = Wav2Vec2ProcessorWithLM.from_pretrained('IbrahimSalah/Syllables_final_Large')
model = Wav2Vec2ForCTC.from_pretrained("IbrahimSalah/Syllables_final_Large")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
import pandas as pd
dftest = pd.DataFrame(columns=['audio'])
import datasets
from datasets import Dataset
path ='/content/tZLR9sn6VnsjYS5xT2qFB.wav'
dftest['audio']=[path]  ## audio path
dataset = Dataset.from_pandas(dftest)


In [4]:
dftest

,audio
0,/content/1-1.wav


In [8]:
!pip install soundfile
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.4 MB/s eta 0:00:00


In [ ]:
# import torch
# import torchaudio
# def speech_file_to_array_fn(batch):
#     speech_array, sampling_rate = torchaudio.load(batch["audio"])
#     print(sampling_rate)
#     resampler = torchaudio.transforms.Resample(sampling_rate, 16_000) # The original data was with 48,000 sampling rate. You can change it according to your input.
#     batch["audio"] = resampler(speech_array).squeeze().numpy()
#     return batch


import torch
import torchaudio
import soundfile as sf  

def speech_file_to_array_fn(batch):
    speech_array, sampling_rate = sf.read(batch["audio"])

    # Convert Numpy array to Pytorch Tensor
    speech_array = torch.from_numpy(speech_array).float()

    if speech_array.ndim == 1:
        speech_array = speech_array.unsqueeze(0)
    else:
        # (Channels, Time)
        speech_array = speech_array.t()

    print(sampling_rate)

    resampler = torchaudio.transforms.Resample(sampling_rate, 16_000)
    batch["audio"] = resampler(speech_array).squeeze().numpy()
    return batch


In [ ]:
import numpy as np
import torch

# 1. استخراج الصوت يدوياً لضمان أنه بصيغة قائمة (List) وليس Column
# نقوم بتحويل كل عنصر إلى مصفوفة Numpy لضمان التوافق
audio_input = [np.array(x) for x in test_dataset["audio"]]

# 2. التأكد من أن الأبعاد صحيحة (يجب أن تكون مصفوفة أحادية البعد لكل ملف صوتي)
# إذا كانت المصفوفة ثنائية الأبعاد (مثلاً 1xN)، نقوم بضغطها لتصبح N فقط
audio_input = [x.squeeze() if x.ndim > 1 else x for x in audio_input]

print(f"Processed audio shape: {audio_input[0].shape}")

# 3. تمرير البيانات المعالجة إلى الـ Processor
inputs = processor(audio_input, sampling_rate=16_000, return_tensors="pt", padding=True)

# 4. التنبؤ (Prediction)
with torch.no_grad():
    logits = model(inputs.input_values).logits
    print("Logits shape:", logits.numpy().shape)

# 5. فك التشفير (Decoding)
transcription = processor.batch_decode(logits.numpy())[0]
print("\nPrediction:", transcription)

Processed audio shape: (121115,)
Logits shape: (1, 378, 3086)

Prediction: ['ءُوْ لَاْ ءِ كَ لَمْ يَ كُوْ نُ مُعْ جِ دِيْ نَ فِلْ ءَ رِضْ وَ مَاْ كَاْ نَ لَ ھُمْ مِنْ ءَوْ لِ يَهْ']


In [17]:
test_dataset

Dataset({
    features: ['audio'],
    num_rows: 1
})

In [18]:
inputs

{'input_values': tensor([[ 1.4882e-06,  1.4882e-06,  1.4882e-06,  ..., -4.1790e-02,
         -3.5248e-02, -5.3419e-02]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32)}